In [1]:
# Load env variables and create client
from dotenv import load_dotenv
import os 
from anthropic import Anthropic

load_dotenv()

client = Anthropic()
model = os.environ["CLAUDE_MODEL"] 

In [2]:
# Helper functions
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)


def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)


def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message.content[0].text

In [7]:
import json

def generate_dataset():  
    prompt = """
        Generate a evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts
        that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects,
        each representing task that requires Python, JSON, or a Regex to complete.

        Example output:
        ```json
        [
            {
                "task": "Description of task",
            },
            ...additional
        ]
        ```

        * Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a regular expression.
        * Focus on tasks that do not require writing much code

        Please generate 3 objects.
    """

    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```json") 
    text = chat(messages, stop_sequences= ["```"]) 
    return json.loads(text) 

In [8]:
dataset = generate_dataset() 
dataset

[{'task': "Write a Python function that parses an AWS ARN string and returns a dictionary containing the partition, service, region, account-id, and resource components. For example, 'arn:aws:s3:us-east-1:123456789012:bucket/my-bucket' should return {'partition': 'aws', 'service': 's3', 'region': 'us-east-1', 'account_id': '123456789012', 'resource': 'bucket/my-bucket'}."},
 {'task': "Create a JSON object representing an AWS S3 bucket policy that allows public read access (s3:GetObject) to all objects within a bucket named 'my-public-assets', but only for objects under the 'images/' prefix."},
 {'task': 'Write a regular expression that validates AWS IAM user names according to AWS requirements: must be between 1-64 characters long, and can only contain alphanumeric characters plus these special characters: plus (+), equal (=), comma (,), period (.), at (@), underscore (_), and hyphen (-).'}]

In [9]:
with open("dataset.json", "w") as f:
    json.dump(dataset, f, indent = 2) 

In [10]:
def run_prompt(test_case):
    """Merges the prompt and test case input, then returns the result"""
    prompt = f"""
        Please solve the following task:

        {test_case["task"]} 
    """
    
    messages = []
    add_user_message(messages, prompt)
    output = chat(messages) 
    return output 

In [11]:
def run_test_case(test_case):
    """Calls run_prompt, then grades the result"""
    output = run_prompt(test_case)
    
    # TODO - Grading
    score = 10
    
    return {
        "output": output,
        "test_case": test_case,
        "score": score
    } 

In [12]:
def run_eval(dataset):
    """Loads the dataset and calls run_test_case with each case"""
    results = []
    
    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)
    
    return results

In [13]:
with open("dataset.json", "r") as f:
    dataset = json.load(f) 

results = run_eval(dataset) 

In [14]:
results 

[{'output': '```python\ndef parse_arn(arn_string):\n    """\n    Parse an AWS ARN string and return a dictionary with its components.\n    \n    ARN format: arn:partition:service:region:account-id:resource\n    \n    Args:\n        arn_string (str): The ARN string to parse\n        \n    Returns:\n        dict: Dictionary containing partition, service, region, account_id, and resource\n        \n    Example:\n        >>> parse_arn(\'arn:aws:s3:us-east-1:123456789012:bucket/my-bucket\')\n        {\'partition\': \'aws\', \'service\': \'s3\', \'region\': \'us-east-1\', \n         \'account_id\': \'123456789012\', \'resource\': \'bucket/my-bucket\'}\n    """\n    # Split the ARN by colons\n    parts = arn_string.split(\':\', 5)  # Split into maximum 6 parts\n    \n    # Validate that we have an ARN\n    if len(parts) != 6 or parts[0] != \'arn\':\n        raise ValueError(f"Invalid ARN format: {arn_string}")\n    \n    # Create dictionary with the components\n    return {\n        \'partiti

In [15]:
print(json.dumps(results, indent = 2))

[
  {
    "output": "```python\ndef parse_arn(arn_string):\n    \"\"\"\n    Parse an AWS ARN string and return a dictionary with its components.\n    \n    ARN format: arn:partition:service:region:account-id:resource\n    \n    Args:\n        arn_string (str): The ARN string to parse\n        \n    Returns:\n        dict: Dictionary containing partition, service, region, account_id, and resource\n        \n    Example:\n        >>> parse_arn('arn:aws:s3:us-east-1:123456789012:bucket/my-bucket')\n        {'partition': 'aws', 'service': 's3', 'region': 'us-east-1', \n         'account_id': '123456789012', 'resource': 'bucket/my-bucket'}\n    \"\"\"\n    # Split the ARN by colons\n    parts = arn_string.split(':', 5)  # Split into maximum 6 parts\n    \n    # Validate that we have an ARN\n    if len(parts) != 6 or parts[0] != 'arn':\n        raise ValueError(f\"Invalid ARN format: {arn_string}\")\n    \n    # Create dictionary with the components\n    return {\n        'partition': parts[